# Hosted agents on Microsoft Foundry

Deploy a **Hosted Agent** to Microsoft Foundry using the Microsoft Agent Framework.

This lab deploys a **Contoso Wealth Knowledge Expert** hosted agent into **Team Alpha's spoke**
(`rg-foundry-spoke-alpha-{suffix}`), reusing the existing Foundry account and project. 

The only new resource created is an Azure Container Registry (ACR) for the
agent container image.

The agent answers two kinds of questions: general wealth and asset management concepts
(performance metrics, mandate types, fund vehicle structures), and Contoso Wealth's own
fictional product line (service tiers, mandates, in-house fund families). Both are answered
purely from the agent's system prompt - there are no tools, no knowledge base, no RAG. See
[`08-03-00-hosted-agents.md`](08-03-00-hosted-agents.md) for the prompt-stuffing pattern
this lab demonstrates and when it stops being enough.

**What this notebook does:**
1. Reads spoke credentials from the repo `.env` (populated by earlier project setup labs)
2. Creates an ACR to store the agent container image (as defined by the Dockerfile, main.py, and requirements.txt)
3. Provisions a Capability Host (Azure Container Apps) on the existing spoke account
4. Builds and pushes the agent container to ACR
5. Registers the agent and runs a test conversation

A **Capability Host** is a managed Azure Container Apps environment, attached to your
Foundry account, that provides the compute infrastructure for running hosted agents.
It handles container scheduling, scaling, and networking so your agent container can
receive and respond to requests without you managing the underlying infrastructure.

## Prerequisites

1. **Python environment**: Run `uv sync` from the repository root to create the
   shared `.venv`, then select the `.venv` kernel in VS Code.
2. **`.env` file**: Must be populated by the `05-foundry-project-pattern-setup` labs:
   - `ALPHA_FOUNDRY_ACCOUNT` - spoke account name (set by the project spoke deployment)
   - `ALPHA_FOUNDRY_PROJECT_ENDPOINT` - spoke project endpoint (set by the project spoke deployment)
   - `GATEWAY_URL` - APIM gateway URL (set by the core gateway deployment)
   - `ALPHA_GATEWAY_KEY` - Team Alpha APIM subscription key (set by the core gateway deployment)
   - `CHAT_MODEL` - chat model deployment name, e.g. `gpt-4.1-mini` (set by the core gateway deployment)
3. **Azure CLI**: Run `az login` before executing the cells.
4. **Permissions**: Your identity needs **Owner** or **Contributor** + **User Access Administrator**
   on `rg-foundry-spoke-alpha-{suffix}` to create RBAC assignments.
5. **Docker**: Required for building the agent container image (Step 5).

## Step 1: Configuration

In [1]:
import os, subprocess, hashlib, json, base64, time, requests
from pathlib import Path
from dotenv import load_dotenv

repo_root = Path(subprocess.run(
    'git rev-parse --show-toplevel', shell=True, capture_output=True, text=True
).stdout.strip())
load_dotenv(repo_root / '.env', override=True)

# Suffix - first 6 chars of SHA-256 of subscription ID (matches the core gateway and project spoke naming)
SUB_ID          = subprocess.run('az account show --query id -o tsv', shell=True, capture_output=True, text=True).stdout.strip()
SUBSCRIPTION_ID = SUB_ID
SUFFIX          = hashlib.sha256((SUB_ID + 'v2').encode()).hexdigest()[:6]
TEAM_NAME       = "alpha"
LOCATION        = "eastus2"

# Hosted agent resources deploy into the existing Spoke Alpha resource group (project spoke deployment)
SPOKE_RG = f"rg-foundry-spoke-{TEAM_NAME}-{SUFFIX}"

# Existing Spoke Alpha account and project (from the project spoke deployment - already in .env)
ACCOUNT_NAME     = os.environ["ALPHA_FOUNDRY_ACCOUNT"]
PROJECT_ENDPOINT = os.environ["ALPHA_FOUNDRY_PROJECT_ENDPOINT"]

# ACR name - only new resource created by this lab
ACR_NAME   = f"acr{TEAM_NAME}{SUFFIX}"   # lowercase alphanumeric, globally unique
AGENT_NAME = "contoso-wealth-expert-agent"

# APIM gateway credentials from .env (set by the core gateway deployment) - injected as container env vars in Step 6
GATEWAY_URL = os.environ["GATEWAY_URL"]        # https://apim-foundry-{suffix}.azure-api.net/openai
GATEWAY_KEY = os.environ["ALPHA_GATEWAY_KEY"]  # Team Alpha APIM subscription key
CHAT_MODEL  = os.environ["CHAT_MODEL"]         # e.g. gpt-4.1-mini

print(f"Suffix:           {SUFFIX}")
print(f"Resource Group:   {SPOKE_RG}")
print(f"Account:          {ACCOUNT_NAME}")
print(f"Project Endpoint: {PROJECT_ENDPOINT}")
print(f"ACR Name:         {ACR_NAME}")
print(f"Agent Name:       {AGENT_NAME}")
print(f"Gateway URL:      {GATEWAY_URL}")
print(f"Chat Model:       {CHAT_MODEL}")

Suffix:           c2676f
Resource Group:   rg-foundry-spoke-alpha-c2676f
Account:          aif-spoke-alpha-c2676f
Project Endpoint: https://aif-spoke-alpha-c2676f.services.ai.azure.com/api/projects/project-alpha-c2676f
ACR Name:         acralphac2676f
Agent Name:       contoso-wealth-expert-agent
Gateway URL:      https://apim-foundry-c2676f.azure-api.net/openai
Chat Model:       gpt-4.1-mini


## Step 2: Create agent code

In [2]:
AGENT_DIR = os.path.join(os.getcwd(), "contoso-wealth-agent")
os.makedirs(AGENT_DIR, exist_ok=True)

# main.py - runs inside the hosted agent container
# GATEWAY_URL / GATEWAY_KEY / CHAT_MODEL are injected as container environment variables
main_py = f'''
import os
from agent_framework import Agent
from agent_framework_foundry import FoundryChatClient
from agent_framework_foundry_hosting import ResponsesHostServer
from azure.identity import DefaultAzureCredential

def main():
    project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT") or os.getenv("PROJECT_ENDPOINT")
    # Model takes the form "<connection-name>/<deployment>" so Foundry resolves
    # it through the spoke project's APIM connection (the spoke has no local
    # model deployments - all inference is via the core gateway).
    chat_model       = os.getenv("CHAT_MODEL", "core-alpha/gpt-4.1-mini")

    # FoundryChatClient routes through Foundry's per-project Responses API endpoint
    # using the container's managed identity. No outbound APIM call from the
    # container - Foundry's hosted compute network can't reach arbitrary
    # *.azure-api.net hosts directly.
    chat_client = FoundryChatClient(
        project_endpoint=project_endpoint,
        model=chat_model,
        credential=DefaultAzureCredential(),
        allow_preview=True,
    )

    agent = Agent(
        chat_client,
        name="{AGENT_NAME}",
        id="{AGENT_NAME}",
        instructions="""
        You are Contoso Wealth\'s knowledge expert - an in-house assistant for
        client-service teams at Contoso Wealth, the private banking and wealth
        management division of Contoso Private Investments.

        You answer two kinds of questions.

        1. Wealth and asset management concepts.
           Explain industry terms and ideas precisely and concisely:
           - Performance metrics: time-weighted vs money-weighted return,
             Sharpe and Sortino ratios, alpha, beta, drawdown, MTD / YTD / ITD.
           - Risk concepts: volatility, value-at-risk, diversification, correlation.
           - Allocation: strategic vs tactical asset allocation, rebalancing,
             glide paths.
           - Fund vehicle structures: UCITS, SICAV, FCP, ETF, mutual fund.
           - Reporting and governance: GIPS compliance, the role of an
             Investment Policy Statement (IPS), the difference between
             discretionary, advisory and execution-only mandates.

        2. Contoso Wealth\'s own product line.
           You can describe these services and funds by name:

           Service tiers (by minimum relationship size):
           - Contoso Wealth Essentials - entry tier, CHF 500K minimum.
           - Contoso Wealth Private - core relationship, CHF 2M minimum.
           - Contoso Wealth Premium - UHNW tier, CHF 25M minimum.
           - Contoso Family Office - full multi-generational family office,
             CHF 100M minimum.

           Mandate types:
           - Contoso Discretionary - bank manages the portfolio against an agreed IPS.
           - Contoso Advisory - bank proposes; client confirms each trade.
           - Contoso Custody - execution-only; no advice.

           In-house fund families:
           - Contoso Core - passive, index-tracking funds across major asset classes.
           - Contoso Active Equity - actively-managed equity strategies, regional and global.
           - Contoso Income - fixed-income strategies, investment grade and high yield.
           - Contoso Sustainable - ESG-screened versions of the above.
           - Contoso Alternatives - hedge fund and private-market access for
             qualified investors.

           Thematic strategies:
           - Contoso Climate Solutions, Contoso Healthcare Innovation,
             Contoso Digital Economy.

        Style: friendly, professional, concise. Match the user\'s language
        (English, French, German, or Italian).

        Important boundaries:
        - You explain concepts and describe Contoso products. You do not
          recommend specific investments to specific clients, give tax advice,
          or give regulatory advice. For client-specific questions, direct the
          user to their relationship manager.
        - Do not invent Contoso products that aren\'t in the list above. If
          asked about a product not listed, say so plainly.
        - If a user asks something outside wealth and asset management,
          politely redirect.
        """
    )

    ResponsesHostServer(agent).run()

if __name__ == "__main__":
    main()
'''

# Container SDKs: azure-ai-projects pinned to 2.0.0b4 (highest the
# transitively-required agent-framework-core supports - 2.1.0 has a hard
# resolution conflict with agent-framework). agentserver-agentframework
# bumped to 1.0.0b17 (latest) so the runtime serves the protocol Foundry's
# control plane currently expects.
requirements = '''azure-ai-agentserver-agentframework==1.0.0b17
azure-ai-projects==2.0.0b4
openai>=1.0.0
azure-identity>=1.15.0'''

dockerfile = '''FROM python:3.12-slim
WORKDIR /app
RUN apt-get update && apt-get install -y curl ca-certificates && rm -rf /var/lib/apt/lists/*
COPY requirements.txt ./
RUN pip install --no-cache-dir -r requirements.txt
COPY main.py ./
EXPOSE 8088
CMD ["python", "main.py"]'''

for name, content in [("main.py", main_py), ("requirements.txt", requirements), ("Dockerfile", dockerfile)]:
    with open(os.path.join(AGENT_DIR, name), "w") as f:
        f.write(content.strip())

print(f"Agent files created in {AGENT_DIR}")
for f in os.listdir(AGENT_DIR):
    path = os.path.join(AGENT_DIR, f)
    print(f"  {f} ({os.path.getsize(path)} bytes)")

Agent files created in <repo-root>/08-agents/08-03-hosted-agents/contoso-wealth-agent
  requirements.txt (109 bytes)
  Dockerfile (257 bytes)
  main.py (3934 bytes)


## Step 3: Deploy infrastructure

Deploys into the existing **Spoke Alpha resource group** (`rg-foundry-spoke-alpha-{suffix}`)
created in project setup section. One new resource is added:

| Resource | Name | Purpose |
|----------|------|---------|
| ACR | `acralpha{suffix}` | Container registry for agent images |

The existing spoke account (`aif-spoke-alpha-{suffix}`) and project (`project-alpha-{suffix}`)
are referenced to grant the project identity AcrPull access.

> Takes approximately 1-2 minutes.

In [3]:
# Hosted agent infrastructure deploys into the existing Spoke Alpha resource group
result = subprocess.run(
    f'az group show -n "{SPOKE_RG}" --query name -o tsv',
    shell=True, capture_output=True, text=True
)
if result.returncode != 0:
    raise RuntimeError(
        f"Resource group '{SPOKE_RG}' not found. Complete earlier project setup"
    )
print(f"Resource group: {SPOKE_RG}")

Resource group: rg-foundry-spoke-alpha-c2676f


In [4]:
# Get principal ID from JWT token (avoids graph.microsoft.com network call)
token   = subprocess.run('az account get-access-token --query accessToken -o tsv', shell=True, capture_output=True, text=True).stdout.strip()
payload = token.split('.')[1] + '=='
PRINCIPAL_ID = json.loads(base64.b64decode(payload))['oid']
print(f"Principal ID: {PRINCIPAL_ID}")

print(f"Deploying ACR infrastructure (~1-2 min)...")
!az deployment group create -g "{SPOKE_RG}" --template-file main.bicep \
    -p suffix="{SUFFIX}" \
    -p teamName="{TEAM_NAME}" \
    -p location="{LOCATION}" \
    -p deployerPrincipalId="{PRINCIPAL_ID}" \
    -o table

Principal ID: 5db0aa5d-f281-47a3-9720-04727dec61e8
Deploying ACR infrastructure (~1-2 min)...
=A new Bicep release is available: v0.43.8. Upgrade now by running "az bicep upgrade".
=<repo-root>/08-agents/08-03-hosted-agents/main.bicep(41,9) : Warning BCP334: The provided value can have a length as small as 3 and may be too short to assign to a target with a configured minimum length of 5. [https://aka.ms/bicep/core-diagnostics#BCP334]

Name    State      Timestamp                         Mode         ResourceGroup
------  ---------  --------------------------------  -----------  -----------------------------
main    Succeeded  2026-05-10T13:04:55.375061+00:00  Incremental  rg-foundry-spoke-alpha-c2676f


In [5]:
r = subprocess.run(
    f'az deployment group show -g "{SPOKE_RG}" -n main --query properties.outputs -o json',
    shell=True, capture_output=True, text=True
)
out = json.loads(r.stdout)

ACR_LOGIN_SERVER = out['acrLoginServer']['value']
print(f"ACR Login Server: {ACR_LOGIN_SERVER}")

# Persist ACR login server to .env
env_file = repo_root / '.env'
existing = {}
if env_file.exists():
    for line in env_file.read_text().splitlines():
        if '=' in line and not line.startswith('#'):
            k, _, v = line.partition('=')
            existing[k.strip()] = v.strip()

existing['ALPHA_HOSTED_ACR_LOGIN_SERVER'] = ACR_LOGIN_SERVER
env_file.write_text('\n'.join(f'{k}={v}' for k, v in existing.items()) + '\n')
print(f"ALPHA_HOSTED_ACR_LOGIN_SERVER saved to .env")

ACR Login Server: acralphac2676f.azurecr.io
ALPHA_HOSTED_ACR_LOGIN_SERVER saved to .env


## Step 4: Create capability host (powered by Azure Container Apps)

In [6]:
ACCESS_TOKEN = subprocess.run(
    'az account get-access-token --resource https://management.azure.com/ --query accessToken -o tsv',
    shell=True, capture_output=True, text=True
).stdout.strip()

headers = {"Content-Type": "application/json", "Authorization": f"Bearer {ACCESS_TOKEN}"}

capability_host_url = (
    f"https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}"
    f"/resourceGroups/{SPOKE_RG}"
    f"/providers/Microsoft.CognitiveServices/accounts/{ACCOUNT_NAME}"
    f"/capabilityHosts/agents?api-version=2025-10-01-preview"
)

payload = {"properties": {"capabilityHostKind": "Agents", "enablePublicHostingEnvironment": True}}

print("Creating Capability Host...")
response = requests.put(capability_host_url, headers=headers, json=payload)
print(f"Status: {response.status_code}")

if response.status_code not in [200, 201, 409]:
    print(response.text)

Creating Capability Host...
Status: 201


In [7]:
print("Waiting for Capability Host (2-5 min)...")
for i in range(30):
    response = requests.get(capability_host_url, headers=headers)
    if response.status_code == 200:
        state = response.json().get('properties', {}).get('provisioningState', 'Unknown')
        print(f"  [{i*10}s] {state}")
        if state == 'Succeeded':
            print("\nCapability Host ready.")
            break
        elif state == 'Failed':
            print("\nCapability Host provisioning failed.")
            print(response.json())
            break
    time.sleep(10)
else:
    print("\nTimeout waiting for Capability Host.")

Waiting for Capability Host (2-5 min)...
  [0s] Succeeded

Capability Host ready.


## Step 5: Build and push container

In [8]:
# Build and push to ACR (must use linux/amd64 platform)
print(f"Building {AGENT_NAME}:latest for linux/amd64...")
!az acr build --registry "{ACR_NAME}" --image "{AGENT_NAME}:latest" --platform linux/amd64 ./contoso-wealth-agent/

Building contoso-wealth-expert-agent:latest for linux/amd64...
Packing source code into tar to upload...
Uploading archived source code from '/tmp/build_archive_0513c166b7684e9f9b563a72ee87bb2e.tar.gz'...
Sending context (2.338 KiB) to registry: acralphac2676f...
Queued a build with ID: ch3
Waiting for an agent...
2026/05/10 13:05:26 Downloading source code...
2026/05/10 13:05:27 Finished downloading source code
2026/05/10 13:05:27 Using acb_vol_def3c675-dcb8-438a-b726-37b613e7a5aa as the home volume
2026/05/10 13:05:27 Setting up Docker configuration...
2026/05/10 13:05:28 Successfully set up Docker configuration
2026/05/10 13:05:28 Logging in to registry: acralphac2676f.azurecr.io
2026/05/10 13:05:28 Successfully logged into acralphac2676f.azurecr.io
2026/05/10 13:05:28 Executing step ID: build. Timeout(sec): 28800, Working directory: '', Network: ''
2026/05/10 13:05:28 Scanning for dependencies...
2026/05/10 13:05:29 Successfully scanned dependencies
2026/05/10 13:05:29 Launching co

## Step 6: Register agent

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    HostedAgentDefinition,
    ProtocolVersionRecord,
    AgentProtocol,
)
from azure.identity import DefaultAzureCredential

# Refreshed Foundry hosted-agent preview (April 2026): registers via
# HostedAgentDefinition (azure-ai-projects 2.1.0+, requires allow_preview=True
# on the client), with ProtocolVersionRecord version="1.0.0". The older
# ImageBasedHostedAgentDefinition + version="v1" pairing targeted the
# now-retired initial preview backend.
client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
    allow_preview=True,
)

CONTAINER_IMAGE = f"{ACR_LOGIN_SERVER}/{AGENT_NAME}:latest"

agent = client.agents.create_version(
    agent_name=AGENT_NAME,
    definition=HostedAgentDefinition(
        container_protocol_versions=[ProtocolVersionRecord(protocol=AgentProtocol.RESPONSES, version="1.0.0")],
        cpu="2", memory="4Gi",
        image=CONTAINER_IMAGE,
        environment_variables={
            "PROJECT_ENDPOINT": PROJECT_ENDPOINT,
            "CHAT_MODEL":       f"{CORE_CONNECTION}/{CHAT_MODEL}",
        }
    )
)

print(f"Agent registered: {agent.name} v{agent.version}")
print(f"Image:            {CONTAINER_IMAGE}")

## Step 7: Grant runtime roles to the agent's per-agent managed identity

At registration time Foundry creates a per-agent managed identity (`instance_identity`). The container needs:
- **AcrPull** on the ACR so Foundry's hosted compute can pull the image
- **Foundry User** on the spoke project so the container can call the project's Responses API (which routes through the `core-alpha` connection to the gateway)

These can't be granted by the bicep (the identity doesn't exist yet at deploy time) so the notebook does it after registration.


In [ ]:
import subprocess, json
from azure.core.rest import HttpRequest

# Fetch the agent's per-agent managed identity (created by Foundry at registration)
v = json.loads(client.send_request(HttpRequest("GET", f"/agents/{AGENT_NAME}/versions/{agent.version}?api-version=v1")).text())
inst_principal = v["instance_identity"]["principal_id"]
print(f"Agent identity: {inst_principal}")

acr_id = subprocess.run(f"az acr show -n {ACR_NAME} --query id -o tsv", shell=True, capture_output=True, text=True).stdout.strip()
project_name = PROJECT_ENDPOINT.rstrip('/').split('/')[-1]
project_id = (
    f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{SPOKE_RG}"
    f"/providers/Microsoft.CognitiveServices/accounts/{ACCOUNT_NAME}/projects/{project_name}"
)

# "Azure AI User" was renamed to "Foundry User"; assign by stable role-definition ID
FOUNDRY_USER_ROLE_ID = "53ca6127-db72-4b80-b1b0-d745d6d5456d"
for role, scope in [("AcrPull", acr_id), (FOUNDRY_USER_ROLE_ID, project_id)]:
    r = subprocess.run(
        f"az role assignment create --assignee-object-id {inst_principal} "
        f"--assignee-principal-type ServicePrincipal --role '{role}' --scope '{scope}'",
        shell=True, capture_output=True, text=True
    )
    if r.returncode == 0:
        print(f"  granted: {role}")
    elif "already exist" in r.stderr.lower():
        print(f"  already granted: {role}")
    else:
        print(f"  failed: {role} \u2014 {r.stderr.strip()[:200]}")

print("\nWait ~60s for RBAC propagation, then run Step 8.")


## Step 8: Test agent

In [ ]:
# Refreshed Foundry preview (April 2026): hosted agents are invoked through a
# per-agent OpenAI client obtained via project.get_openai_client(agent_name=...),
# which routes through the agent's hosted endpoint at
# {project_endpoint}/agents/{agent_name}/endpoint/protocols/openai. The older
# extra_body={"agent_reference": ...} pattern is retired.
agent_openai_client = client.get_openai_client(agent_name=AGENT_NAME)

def ask(query: str):
    print(f"\U0001f9d1 {query}\n")
    try:
        response = agent_openai_client.responses.create(input=query)
        print(f"\U0001f916 {response.output_text}")
    except Exception as e:
        print(f"\u274c {e}")

ask("Explain the difference between time-weighted return and money-weighted return, and when each one matters.")


In [ ]:
ask("What's the difference between Contoso Discretionary and Contoso Advisory mandates, and at what relationship size does each become available?")

## Cleanup

In [13]:
# Uncomment to delete the ACR from the Spoke Alpha resource group.
# The core spoke account and project (project spoke deployment) are not affected.
#
# !az acr delete -g "{SPOKE_RG}" -n "{ACR_NAME}" --yes
# print(f"Deleted ACR {ACR_NAME} from {SPOKE_RG}")